# 4.4 AirSim Depth Camera

In a real indoor scenario, a drone can perform 3 types of actions:
1. Look around to see what objects are in the room.
2. Detect specific objects and determine their approximate positions.
3. After detection, navigate toward the target, which involves determining the target's direction and distance from the drone.

These three actions correspond to 3 categories of multimodal LLM capabilities:
1. Image understanding
2. Object detection
3. Object localization

Object localization can also be done by LLMs, but using a depth camera is more practical and accurate.

We've already covered image understanding and object detection in previous lessons. This lesson focuses on object localization.

In previous lessons, we also used LiDAR sensors for distance measurement.

In previous lessons, if we needed to get an object's position, we could:
1. Use the get_position API to get the target's coordinates, then fly_to it.
2. Use object detection to get the target's bounding box position, then use PID control to approach it (not very precise).

A better method:
3. Use a depth camera to get the target's distance and direction information.

## Stereo Vision Principles
---

### Principle: Triangulation and Disparity
1. **Disparity Calculation**
Two cameras (with baseline **B**) capture the same scene from different angles. The same point appears at different pixel positions in the two images (this difference is called Disparity). **The relationship between disparity and distance**: closer objects have larger disparity; farther objects have smaller disparity.

2. **Depth Calculation**
Given camera focal length **f** and baseline **B**, depth **Z** (distance) can be calculated:


Z = B * f / Disparity

For example, with baseline 10 cm, focal length 500 pixels, and disparity 50 pixels: Z = 10 x 500 / 50 = 100 cm


<img src="img/doubles.png" width="640px" />

---

### Key Steps
1. **Camera Calibration**
Determine camera **intrinsic parameters** (focal length, principal point) and **extrinsic parameters** (relative position and orientation between cameras). This is done using calibration tools.

2. **Image Rectification**
Through rectification, align the two images so that corresponding points lie on the same horizontal line, simplifying the search for correspondences.

3. **Stereo Matching**
Find corresponding points between the two images (edges, features), using methods including Block Matching or feature-based approaches (SIFT, ORB) for precise depth calculation.


---

### Errors and Limitations
1. **Common Error Sources**:
- **Installation errors**: Camera misalignment or angle offset
- **Calibration errors**: Imprecise calibration
- **Matching errors**: Incorrect correspondences (e.g., in PSVR)
- **Pixel errors**: Image resolution limits calculation precision

2. **Application Limitations**:
- **Distance limitations**: Effective within about **2 meters**; precision decreases with distance
- **Environment sensitivity**: Textureless scenes or repetitive patterns cause matching failures
- **Motion blur**: Fast movement affects image stability

---

### Application Scenarios
1. **Autonomous Driving**
Obstacle detection, distance measurement (e.g., Microvision), high-precision ranging.
2. **Robotics**
Navigation, AGV obstacle avoidance, path planning.
3. **Virtual Reality (VR)**
PSVR and similar devices use stereo cameras for 6-DOF tracking.
4. **Industrial Inspection**
Short-range measurement (complementing LiDAR and other sensors).

---

## Depth Camera in AirSim

The drone's depth camera, combined with multimodal LLM + depth sensing, enables object localization.

The multimodal LLM identifies objects and provides bounding boxes, while the depth camera uses the bbox coordinates to retrieve distance information, achieving full localization.

AirSim's depth simulation is not based on actual stereo cameras -- it generates pixel-level depth maps directly through Unreal Engine, without stereo matching or disparity calculation.

```python

# AirSim depth camera settings

{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "CameraDefaults": {
        "CaptureSettings": [
            {
                "ImageType": 0,
                "Width": 1280,
                "Height": 720,
                "FOV_Degrees": 90
            },
            {
                "ImageType": 1,
                "Width": 1280,
                "Height": 720,
                "FOV_Degrees": 90
            },
            {
                "ImageType": 2,
                "Width": 1280,
                "Height": 720,
                "FOV_Degrees": 90
            }
        ]
    }
}

```

In the settings above, CaptureSettings ImageType 0, 1, 2 correspond to: Scene (RGB image), DepthPlanar (pixel values represent distance to image plane), and DepthPerspective (pixel values represent distance to camera).

The depth map distance representation is shown below:

<img src='img/deepimg.png' width='640px' />

In [ ]:
from airsim_smol_wrapper import *

# Take off
takeoff()

In [ ]:
set_yaw(90)

In [ ]:
"""
Get camera images from the drone
"""
import sys
sys.path.append('..')
import airsim
client = airsim.MultirotorClient()#run in some machine of airsim,otherwise,set ip="" of airsim

In [ ]:
# Note: First run may be slow due to initialization
responses = client.simGetImages([
    airsim.ImageRequest("0", airsim.ImageType.DepthPerspective, pixels_as_float=True)
])
depth_data = responses[0].image_data_float # float array

In [ ]:
1280*720

In [ ]:
len(depth_data)

In [ ]:
import numpy as np
depth_array = np.array(depth_data).reshape(responses[0].height, responses[0].width)
depth_normalized = (depth_array / depth_array.max() * 255).astype(np.uint8)

import matplotlib.pyplot as plt
# Set up inline display
%matplotlib inline
plt.figure(figsize=(10, 8))

# Display depth map in grayscale
plt.imshow(depth_normalized, cmap='gray', aspect='auto')
plt.axis('off') # Hide axes
plt.title("Normalized Depth Map")
plt.show()

In [ ]:
# Note: First run may be slow due to initialization
responses = client.simGetImages([
    # png format
    airsim.ImageRequest(0, airsim.ImageType.Scene, pixels_as_float=False, compress=True),

    # Floating point uncompressed depth image, pixels represent distance to image plane
    airsim.ImageRequest(0, airsim.ImageType.DepthPlanar, pixels_as_float=True),

    # Pixels represent distance to camera
    airsim.ImageRequest(0, airsim.ImageType.DepthPerspective, pixels_as_float=True)
    ]
)

In [ ]:
img_depth_planar = np.array(responses[1].image_data_float).reshape(responses[1].height, responses[0].width)
img_depth_perspective = np.array(responses[2].image_data_float).reshape(responses[2].height, responses[1].width)

# Regular image
image_data = responses[0].image_data_uint8
img = cv2.imdecode(np.array(bytearray(image_data), dtype='uint8'), cv2.IMREAD_UNCHANGED)  # Read from binary image data
img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)  # 4 channels to 3

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RGBimage
axes[0].imshow(img)
axes[0].set_title("RGB Scene")
axes[0].axis('off')

# depth map (DepthPlanar) 
planar_plot = axes[1].imshow(img_depth_planar, cmap='jet')
plt.colorbar(planar_plot, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_title("DepthPlanar (mm)")
axes[1].axis('off')

# depth map (DepthPerspective) 
perspective_plot = axes[2].imshow(img_depth_perspective, cmap='viridis')
plt.colorbar(perspective_plot, ax=axes[2], fraction=0.046, pad=0.04)
axes[2].set_title("DepthPerspective (mm)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Overlay visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# DepthPlanar overlay
axes[0].imshow(img)
overlay_planar = axes[0].imshow(img_depth_planar, cmap='jet', alpha=0.5)
plt.colorbar(overlay_planar, ax=axes[0], label='Distance (mm)')
axes[0].set_title("RGB + DepthPlanar Overlay")
axes[0].axis('off')

# DepthPerspective overlay
axes[1].imshow(img)
overlay_perspective = axes[1].imshow(img_depth_perspective, cmap='viridis', alpha=0.5)
plt.colorbar(overlay_perspective, ax=axes[1], label='Distance (mm)')
axes[1].set_title("RGB + DepthPerspective Overlay")
axes[1].axis('off')

plt.tight_layout()
plt.show()